# LIBRARY

In [22]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from sklearn.preprocessing import MinMaxScaler
import json
import os
import pandas as pd

def distribution_similarity(real_traffic, gen_traffic, bins=10):

    # Normalization and Histogram
    real_hist, _ = np.histogram(real_traffic, bins=bins, density=True)
    gen_hist, _ = np.histogram(gen_traffic, bins=bins, density=True)

    # Prevent zeros by adding lower case numbers
    """Jensen-Shannon Divergence can have problems with histograms containing zeros (log(0) is undefined). 
    Therefore, a small number (1e-9) is added to the histograms to remove the zeros."""
    real_hist += 1e-9
    gen_hist += 1e-9

    # Similarity Calculation
    """Jensen-Shannon Divergence (JSD): Measures the similarity between two probability distributions 
    (0: exactly the same, 1: completely different)."""
    js = jensenshannon(real_hist, gen_hist)

    return js

# RESULTS

In [23]:
real_traffic = []
with open(r'../Datasets/Experiment_1_one_way_communication_10_minute_input_sample.json', 'r') as file:
    for line in file:
        real_traffic.append(json.loads(line))

real_traffic  = pd.DataFrame(real_traffic)
real_traffic = real_traffic.drop(columns=["No.", "Info_clean"])


## EXPERIMENT 1

In [24]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon

N = list(range(0,10)) # number of trial

num_pac = []
pack_size = []
time_mean = []
time_min = []
time_max = []
JS_metric = []

for n in N:
    file_path = f"../Generated_Traffic/JSON_files/exp1_gan_generated_packets/generated_packets_{n}.json"

    with open(file_path, 'r') as f:
        generated_traffic = json.load(f)

    generated_traffic = pd.DataFrame(generated_traffic)
    print(f"\n ================== Generated Traffic Trial Number {n} ==================")      

    print("Number of Packet:", len(generated_traffic))
    print("Average Packet Size:", (generated_traffic['length'].astype(int)).mean())
    print("Time Interval Average:", (generated_traffic['time'].astype(float)).diff().mean())
    
    print("Time Interval min:", (generated_traffic['time'].astype(float)).diff().min())
    print("Time Interval max:", (generated_traffic['time'].astype(float)).diff().max())
    
    print("Number of source:", generated_traffic['src'].nunique())
    print("Number of destination:", generated_traffic['dst'].nunique())
    
    num_pac.append(len(generated_traffic))
    pack_size.append((generated_traffic['length'].astype(int)).mean())
    time_mean.append((generated_traffic['time'].astype(float)).diff().mean())
    time_min.append((generated_traffic['time'].astype(float)).diff().min())
    time_max.append((generated_traffic['time'].astype(float)).diff().max())


    """ JENSEN SHANNON FOR TIMESTAMPT"""

    generated_traffic['time'] = pd.to_numeric( generated_traffic['time'], errors='coerce')
    real_traffic['Time'] = pd.to_numeric(real_traffic['Time'], errors='coerce')

    js_packet = distribution_similarity(real_traffic['Time'], generated_traffic['time'] )
    print(f"\nJensen-Shannon for time: {js_packet:.2f}")
    JS_metric.append(js_packet)
    

print("Number of Packet:", np.mean(num_pac),  
    "Average Packet Size", np.mean(pack_size),
    "Time Interval Average", np.mean(time_mean),
    "Time Interval min",  np.mean(time_min),
    "Time Interval max",  np.mean(time_max),
     "Average JS:", np.mean(JS_metric)
    )

print(JS_metric)



 ================== Generated Traffic Trial Number 0 ==================
Number of Packet: 128
Average Packet Size: 59.546875
Time Interval Average: 0.7058787952755906
Time Interval min: 0.0714030000000001
Time Interval max: 12.237541
Number of source: 1
Number of destination: 2

Jensen-Shannon for time: 0.34

 ================== Generated Traffic Trial Number 1 ==================
Number of Packet: 128
Average Packet Size: 59.578125
Time Interval Average: 0.699757748031496
Time Interval min: 0.07182299999999842
Time Interval max: 11.444031000000003
Number of source: 1
Number of destination: 2

Jensen-Shannon for time: 0.34

 ================== Generated Traffic Trial Number 2 ==================
Number of Packet: 128
Average Packet Size: 59.484375
Time Interval Average: 0.693291905511811
Time Interval min: 0.0698609999999995
Time Interval max: 12.001795999999999
Number of source: 1
Number of destination: 2

Jensen-Shannon for time: 0.32

 ================== Generated Traffic Trial Numbe

# EXPERIMENT 2

In [25]:
real_traffic = []
with open(r'../Datasets/Experiment_2_input_sample.json', 'r') as file:
    for line in file:
        real_traffic.append(json.loads(line))

real_traffic  = pd.DataFrame(real_traffic)
real_traffic = real_traffic.drop(columns=["No.", "Info_clean"])


In [26]:
import pandas as pd
import numpy as np
import json
from scipy.spatial.distance import jensenshannon

N = list(range(0,10)) # number of trial

num_pac = []
pack_size = []
time_mean = []
time_min = []
time_max = []
JS_metric = []

for n in N:
    file_path = f"../Generated_Traffic/JSON_files/exp2_gan_generated_packets/generated_packets_{n}.json"

    with open(file_path, 'r') as f:
        generated_traffic = json.load(f)
    generated_traffic = pd.DataFrame(    generated_traffic)

    print(f"\n ================== Generated Traffic Trial Number {n} ==================")      

    print("Number of Packet:", len(generated_traffic))
    print("Average Packet Size:", (generated_traffic['length'].astype(int)).mean())
    print("Time Interval Average:", (generated_traffic['time'].astype(float)).diff().mean())
    print("Time Interval min:", (generated_traffic['time'].astype(float)).diff().min())
    print("Time Interval max:", (generated_traffic['time'].astype(float)).diff().max())
    print("Number of source:", generated_traffic['src'].nunique())
    print("Number of destination:", generated_traffic['dst'].nunique())
    
    num_pac.append(len(generated_traffic))
    pack_size.append((generated_traffic['length'].astype(int)).mean())
    time_mean.append((generated_traffic['time'].astype(float)).diff().mean())
    time_min.append((generated_traffic['time'].astype(float)).diff().min())
    time_max.append((generated_traffic['time'].astype(float)).diff().max())


    """ JENSEN SHANNON FOR TIMESTAMPT"""

    generated_traffic['time'] = pd.to_numeric( generated_traffic['time'], errors='coerce')
    real_traffic['Time'] = pd.to_numeric(real_traffic['Time'], errors='coerce')


    gen_inter_time = generated_traffic['time'].astype(float).diff()
    real_inter_time = real_traffic['Time'].astype(float).diff()

    js_packet = distribution_similarity(real_traffic['Time'], generated_traffic['time'] )
    # js_packet = distribution_similarity(real_inter_time[1:-1], gen_inter_time[1:-1])
    print(f"\nJensen-Shannon for time: {js_packet:.2f}")
    JS_metric.append(js_packet)
    

print("Number of Packet:", np.mean(num_pac),  
    "Average Packet Size", np.mean(pack_size),
    "Time Interval Average", np.mean(time_mean),
    "Time Interval min",  np.mean(time_min),
    "Time Interval max",  np.mean(time_max),
     "Average JS:", np.mean(JS_metric)
    )

print(JS_metric)




 ================== Generated Traffic Trial Number 0 ==================
Number of Packet: 220
Average Packet Size: 57.71363636363636
Time Interval Average: 0.1988888401826484
Time Interval min: 0.03742600000000351
Time Interval max: 3.3132260000000002
Number of source: 2
Number of destination: 3

Jensen-Shannon for time: 0.19

 ================== Generated Traffic Trial Number 1 ==================
Number of Packet: 220
Average Packet Size: 57.654545454545456
Time Interval Average: 0.1973425844748858
Time Interval min: 0.038499000000001615
Time Interval max: 2.9158540000000013
Number of source: 2
Number of destination: 3

Jensen-Shannon for time: 0.16

 ================== Generated Traffic Trial Number 2 ==================
Number of Packet: 220
Average Packet Size: 57.64090909090909
Time Interval Average: 0.19515983105022833
Time Interval min: 0.03600100000000239
Time Interval max: 2.767650999999999
Number of source: 2
Number of destination: 3

Jensen-Shannon for time: 0.16

 =========